In [2]:
import anndata as ad
import decoupler as dc
import mofax as mfx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import statsmodels.stats.multitest as multitest

In [3]:
PATH_ADATA = "/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad"
# --- Cargar datos ---
adata = ad.read_h5ad(PATH_ADATA)

pesos_consenso = pd.DataFrame(
    adata.uns['mofa_weights'],
    index=adata.uns['mofa_weights_genes'],
    columns=adata.uns['mofa_weights_factors']
)
pesos_t = pesos_consenso.T  # Factores x Genes

factor_order = sorted(pesos_t.index, key=lambda x: int(x.replace("Factor", "")))

print(f"Matriz de pesos: {pesos_consenso.shape} (genes x factores)")
print(f"Factores: {pesos_t.index.tolist()}")

Matriz de pesos: (18775, 30) (genes x factores)
Factores: ['Factor1', 'Factor2', 'Factor3', 'Factor4', 'Factor5', 'Factor6', 'Factor7', 'Factor8', 'Factor9', 'Factor10', 'Factor11', 'Factor12', 'Factor13', 'Factor14', 'Factor15', 'Factor16', 'Factor17', 'Factor18', 'Factor19', 'Factor20', 'Factor21', 'Factor22', 'Factor23', 'Factor24', 'Factor25', 'Factor26', 'Factor27', 'Factor28', 'Factor29', 'Factor30']


In [4]:
collectri = dc.get_collectri(organism='human', top=500)
tfs_of_interes = ['YAP1', 'TEAD1', 'TEAD2', 'TEAD4','WWTR1']
acts,pvals = dc.run_ulm(pesos_t,
                                    net=collectri,
                                    source= 'source',
                                    target = 'target',
                                    weight= 'weight',
                                    )
print(f"Actividades PROGENy: {acts.shape}")
acts_of_interest = acts[tfs_of_interes]
pvals_of_interest = pvals[tfs_of_interes]
acts_of_interest.head()
pvals_of_interest.head()

  File "/mnt/netapp2/Store_uni/home/ulc/co/mao/SOFTWARE/miniconda3/envs/sc_2/lib/python3.11/site-packages/decoupler/omnip.py", line 523, in get_collectri
    ct = op.interactions.CollecTRI.get(
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/netapp2/Store_uni/home/ulc/co/mao/SOFTWARE/miniconda3/envs/sc_2/lib/python3.11/site-packages/omnipath/_core/requests/_utils.py", line 114, in wrapper
    return wrapped(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/netapp2/Store_uni/home/ulc/co/mao/SOFTWARE/miniconda3/envs/sc_2/lib/python3.11/site-packages/omnipath/_core/requests/_utils.py", line 31, in _get_helper
    return cls()._get(**kwargs)
           ^^^^^^^^^^^^^^^^^^^^
  File "/mnt/netapp2/Store_uni/home/ulc/co/mao/SOFTWARE/miniconda3/envs/sc_2/lib/python3.11/site-packages/omnipath/_core/requests/_request.py", line 108, in _get
    kwargs = self._validate_params(kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/netapp2/Store_uni/home/ulc/co/mao/SOFTWA

Actividades PROGENy: (30, 757)


,YAP1,TEAD1,TEAD2,TEAD4,WWTR1
Factor1,0.979588,0.656489,0.246794,0.121180,0.955990
Factor2,0.913006,0.206807,0.971154,0.144510,0.282503
Factor3,0.084851,0.545569,0.035415,0.694903,0.081796
Factor4,0.832271,0.844906,0.662420,0.235518,0.918674
Factor5,0.008624,0.000005,0.641685,0.095346,0.027480


In [5]:
acts_of_interest.columns

Index(['YAP1', 'TEAD1', 'TEAD2', 'TEAD4', 'WWTR1'], dtype='object')

In [6]:
resultados = []  
for factor in acts_of_interest.index:
    for tf in acts_of_interest.columns:
        resultados.append({
                'factor': factor,
                'TF': tf,
                'Actividad': acts_of_interest.loc[factor,tf],
                'pval': pvals_of_interest.loc[factor,tf]
                })


resultados_df = pd.DataFrame(resultados)
mask = resultados_df['pval'].notna()
resultados_df.loc[mask, 'padj'] = multitest.multipletests(resultados_df.loc[mask, 'pval'], method='fdr_bh')[1]
factores_interes = resultados_df[(resultados_df['padj'] < 0.05) & (resultados_df['Actividad'] < 0)]

In [ ]:
resultados_top = []
for _, row in factores_interes.iterrows():
    factor = row['factor']
    tf = row['TF']
    pval = row['pval']
    padj = row['padj']
    top_negative = adata[:, factor].to_df()[[factor]].sort_values(factor).head(10)
    top_negative['factor'] = factor
    top_negative['TF'] = tf
    top_negative['pval'] = pval
    top_negative['padj'] = padj

    top_negative = top_negative.rename(columns={factor : 'score'})
    resultados_top.append(top_negative)

resultados_top_df = pd.concat(resultados_top)

resultados_top_df[['drug', 'concentration']] = resultados_top_df.index.to_series().str.rsplit('_', n=2, expand=True)[[0, 1]].values


In [32]:

resultados_top_df.to_excel('/home/ulc/co/mao/drug_clustering/TFM_mao/resultados/yap_taz/factores_yap_taz.xlsx')


DIGITOXIN_0.5_11                 11
TAK-901_5.0_3                     3
OUABAIN_(OCTAHYDRATE)_5.0_12     12
HARRINGTONINE_0.5_8               8
DINACICLIB_5.0_9                  9
DINACICLIB_0.5_8                  8
DIGITOXIN_5.0_12                 12
MEBENDAZOLE_5.0_9                 9
DINACICLIB_0.05_7                 7
OUABAIN_(OCTAHYDRATE)_0.05_10    10
Name: 2, dtype: object

In [16]:
resultados_top_df.index

Index(['DIGITOXIN_0.5_11', 'TAK-901_5.0_3', 'OUABAIN_(OCTAHYDRATE)_5.0_12',
       'HARRINGTONINE_0.5_8', 'DINACICLIB_5.0_9', 'DINACICLIB_0.5_8',
       'DIGITOXIN_5.0_12', 'MEBENDAZOLE_5.0_9', 'DINACICLIB_0.05_7',
       'OUABAIN_(OCTAHYDRATE)_0.05_10'],
      dtype='object')

In [11]:
factores_interes.columns[0]

'factor'